<a href="https://colab.research.google.com/github/aravindasubramanian-hash/aravind/blob/main/qcommerce_colab_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q-Commerce Last-Mile — Colab Runner

Completes the two blocked inputs (road graph, population) and re-runs the
analysis chain on real data.

**Run cells top to bottom.** Every stage checkpoints to Google Drive, so a
dropped session resumes instead of restarting — Colab disconnects after ~90 min
idle and caps at 12 h, and the solves are slow.

### Before you start
1. Upload `qcommerce_project.zip` when Cell 2 asks.
2. Runtime -> Change runtime type -> **High-RAM** if offered. The road graph step
   is the memory-heavy one.

### Stage map
| Cell | Stage | Est. time | Blocking? |
|---|---|---|---|
| 1-3 | Setup, Drive mount, unpack | 3 min | — |
| 4 | **Road graph** (OSM) | 10-25 min | blocks everything |
| 5 | Routed travel time (control profile) | 15-40 min | blocks 7+ |
| 6 | Population (building footprints) | 10 min | blocks 8 |
| 7 | **V2.1 experiment**: placeholder vs routed | 20-40 min | standalone result |
| 8 | Demand re-run + gate | 5 min | blocks 9 |
| 9 | Facility + SMILP re-run | 60-90 min | — |
| 10 | Store-capacity costing | 20 min | — |

**Gate:** if Cell 4 has not succeeded by end of Day 2, stop and publish on
placeholder geometry with the limitation stated. Half-integrated real data with
no synthesis written is the bad outcome.

## Cell 1 — Install

Pinned to avoid the osmnx 1.x/2.x API break.

In [ ]:
!pip install -q "osmnx>=2.0" geopandas pyomo highspy pyarrow networkx scipy rtree
import osmnx, geopandas, pyomo, networkx, scipy
print("osmnx", osmnx.__version__, "| geopandas", geopandas.__version__)
import pyomo.environ as pyo
print("HiGHS available:", pyo.SolverFactory("appsi_highs").available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.3 MB/s eta 0:00:00
osmnx 2.1.1 | geopandas 1.2.0
HiGHS available: True


## Cell 2 — Mount Drive and unpack

Everything lives in Drive, not on the VM disk. **This is the single most
important cell**: a 10-hour session that dies at hour 9 with results on the VM
loses all of it.

In [ ]:
import shutil, os
src = {'travel_time.py': f'{ROOT}/03_network', 'demand_engine.py': f'{ROOT}/04_demand'}
for fn, d in src.items():
    new = f'/content/{fn}'
    target = f'{d}/{fn}'
    assert os.path.exists(new), new
    if os.path.exists(target) and not os.path.exists(target + '.orig'):
        shutil.copy(target, target + '.orig')   # keep the pre-patch original once
    shutil.copy(new, target)
    print('installed', target, os.path.getsize(target), 'bytes | backup .orig:', os.path.exists(target + '.orig'))
# sanity: patched markers present
print('travel_time patched:', '_off = ~np.eye' in open(f'{ROOT}/03_network/travel_time.py').read())
print('demand_engine patched:', 'mean_adjacent_hour_rank_corr_normalised' in open(f'{ROOT}/04_demand/demand_engine.py').read())


installed /content/drive/MyDrive/qcommerce/03_network/travel_time.py 11847 bytes | backup .orig: True
installed /content/drive/MyDrive/qcommerce/04_demand/demand_engine.py 15406 bytes | backup .orig: True
travel_time patched: True
demand_engine patched: True


In [ ]:
import os, json, time, glob
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/qcommerce'
RES = f'{ROOT}/results'
def done(*paths):
    return all(os.path.exists(p) and os.path.getsize(p) > 0 for p in paths)
def stamp(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")
def show(path, keys=None):
    d = json.load(open(path))
    print(json.dumps({k: d[k] for k in keys} if keys else d, indent=2, default=str)[:2000])
print("ROOT:", os.path.exists(ROOT))
for f in sorted(glob.glob(f'{RES}/w8_*.json')) + sorted(glob.glob(f'{RES}/final_v2r.json')):
    print(os.path.basename(f), time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(f))))
print("footprints:", os.path.exists(f'{ROOT}/02_data/footprints/bengaluru_buildings.geojson'))


Mounted at /content/drive
ROOT: True
w8_cap2000.json 2026-10-11 12:30
w8_cap220.json 2026-10-11 12:30
w8_cap440.json 2026-10-11 12:30
w8_cap880.json 2026-10-11 12:30
w8_free220.json 2026-10-11 12:30
w8_noret220.json 2026-10-11 12:30
w8_real_base.json 2026-10-11 22:15
w8_real_free.json 2026-10-11 23:28
final_v2r.json 2026-10-11 22:09
footprints: True


In [ ]:
for t in ['real_base','real_free']:
    r = json.load(open(f'{RES}/w8_{t}.json'))
    fl = sum(v['rebal_flow'] for v in r['per_scenario'].values())
    print(f"{t:10} cost={r['total_cost']:>14,.0f} expSL={r['expected_sl']:6.2f} flow={fl:>12,.1f} riders={r['total_riders']:,.0f}")
    print("   ", {k: (round(v['sl'],2), round(v['rebal_flow'],1)) for k,v in r['per_scenario'].items()})


real_base  cost=    62,530,553 expSL= 94.47 flow=         0.0 riders=59,515
    {'BAU': (96.82, 0.0), 'WEEKEND': (94.99, 0.0), 'EVENT': (94.55, 0.0), 'MEGA': (63.46, 0.0)}
real_free  cost=    62,529,052 expSL= 94.47 flow=   240,626.8 riders=59,498
    {'BAU': (96.82, 89459.6), 'WEEKEND': (94.99, 73495.2), 'EVENT': (94.55, 61256.9), 'MEGA': (63.45, 16415.1)}


In [ ]:
import glob, os
RES = f'{ROOT}/results'
def done(*paths):
    return all(os.path.exists(p) and os.path.getsize(p) > 0 for p in paths)
print("ROOT exists:", os.path.exists(ROOT), "| routed matrix:", done(f'{RES}/ward_travel_time_369_routed_control.parquet'))
print("VM footprints:", glob.glob('/content/footprints/*'))
hits = []
for pat in ['*uilding*', '*ootprint*']:
    hits += glob.glob(f'/content/drive/MyDrive/**/{pat}', recursive=True)
for h in sorted(set(hits))[:20]:
    print(h, round(os.path.getsize(h)/1e6,1), 'MB')
print("done scanning")


## Cell 3 — Checkpoint helper

`done(path)` lets every later cell skip work already on Drive. Re-running the
whole notebook after a disconnect then costs only the unfinished stages.

In [ ]:
import os, json, time
RES = f'{ROOT}/results'

def done(*paths):
    return all(os.path.exists(p) and os.path.getsize(p) > 0 for p in paths)

def stamp(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

def show(path, keys=None):
    d = json.load(open(path))
    print(json.dumps({k: d[k] for k in keys} if keys else d, indent=2, default=str)[:2000])

NameError: name 'ROOT' is not defined

## Cell 4 — Road graph (BLOCKING)

Downloads the OSM drive network for the GBA area plus a 2 km buffer and runs
five validation checks.

**`network_type="drive"`, not `"bike"`** — delivery riders are on motorcycles and
follow motor-vehicle restrictions. `"bike"` would route them down links they
legally cannot use.

**The buffer matters.** Routing between two edge wards may legitimately leave the
boundary and come back; a tight clip forces a longer interior path and silently
inflates travel times.

**Read G2 and G4 in the output.** G2 (strongly connected core) catches one-way
pockets that would produce infinite travel times later. G4 flags any ward
snapping >250 m from the network — a lake, an enclave, or an extract that missed
it. Send me the printed block.

In [ ]:
%cd {ROOT}/03_network
if done(f'{ROOT}/03_network/blr_drive.graphml', f'{ROOT}/03_network/ward_snap_nodes.csv'):
    stamp("road graph already built - skipping")
else:
    !python3 road_graph.py --wards {ROOT}/02_data/wards_gba_2025_369.gpkg
stamp("done")
show(f'{ROOT}/03_network/road_graph_validation.json')

/content/drive/MyDrive/qcommerce/03_network
  fetching drive network for 0.0821 deg^2 (buffer 2000 m)...
  maxspeed tagged on 9,390 of 472,552 edges (2.0%) — remainder imputed

ROAD GRAPH VALIDATION
  [PASS] G1 Graph size                     188,396 nodes, 472,552 edges
  [PASS] G2 Strongly connected core        largest SCC holds 188,135 of 188,396 nodes (99.86%)
  [PASS] G3 Edge travel_time present       0 edges missing travel_time
  [FAIL] G4 Ward snap distance             median 33 m, max 1502 m; 15 wards beyond 250 m -> [('Central|24', 425.1), ('East|46', 414.0), ('South|29', 278.7), ('East|27', 463.1), ('East|33', 1501.6), ('East|49', 306.0), ('Central|14', 256.8), ('South|37', 260.4)]
  [PASS] G5 Distinct snap nodes per ward   0 wards share a snap node with another ward

  4/5 passed — 1 FAILED

wrote blr_drive.graphml, ward_snap_nodes.csv, road_graph_validation.json
[12:35:59] done
{
  "checks": [
    {
      "id": "G1",
      "name": "Graph size",
      "passed": true,
      "d

In [ ]:
import shutil, numpy as np
SNAP = f'{ROOT}/03_network/ward_snap_nodes.csv'
BAK = f'{ROOT}/03_network/ward_snap_nodes.backup_pre_resnap.csv'
if not os.path.exists(BAK):
    shutil.copy(SNAP, BAK)          # never overwrite the original backup
snap0 = pd.read_csv(BAK)             # always re-snap from the pristine original
print("columns:", list(snap0.columns))

# which column holds the graph node id?
node_ids = set(G.nodes())
node_col = next(c for c in snap0.columns if c not in ('uid','snap_dist_m')
                and snap0[c].isin(node_ids).all())
print("node column:", node_col)

# restrict candidates to the strongly connected core (no one-way pockets)
scc = max(nx.strongly_connected_components(G), key=len)
cand = nodes[nodes.n.isin(scc)].copy()
cand_m = cand.to_crs(32643)
w_m = w.to_crs(32643).set_index('uid')

snap1 = snap0.copy()
used = set(snap1[node_col])
bad = snap0[snap0.snap_dist_m > 250].uid.tolist()
for u in bad:
    poly = w_m.loc[u].geometry
    cen = poly.centroid
    pool = cand_m[cand_m.within(poly)]
    pool = pool[~pool.n.isin(used - {snap1.loc[snap1.uid == u, node_col].iloc[0]})]
    d = pool.geometry.distance(cen)
    best = pool.loc[d.idxmin()]
    old = snap1.loc[snap1.uid == u, node_col].iloc[0]
    snap1.loc[snap1.uid == u, node_col] = best.n
    snap1.loc[snap1.uid == u, 'snap_dist_m'] = round(float(d.min()), 1)
    used.discard(old); used.add(best.n)
    print(f"{u:14} {snap0.loc[snap0.uid==u,'snap_dist_m'].iloc[0]:7.0f} m -> {d.min():6.0f} m   (nodes inside={len(pool)})")

snap1.to_csv(SNAP, index=False)
chg = (snap1[node_col] != snap0[node_col]).sum()
print(f"\nrows changed: {chg} (expected {len(bad)});  wrote {SNAP}")

# --- recheck G4 / G5 ---
over = snap1[snap1.snap_dist_m > 250]
print(f"G4 {'PASS' if len(over)==0 else 'FAIL'}  median {snap1.snap_dist_m.median():.0f} m, max {snap1.snap_dist_m.max():.0f} m, wards >250 m: {len(over)}")
if len(over): print(over[['uid','snap_dist_m']].to_string(index=False))
dups = snap1[node_col].duplicated().sum()
print(f"G5 {'PASS' if dups==0 else 'FAIL'}  duplicate snap nodes: {dups}")
print("all snap nodes in graph:", snap1[node_col].isin(node_ids).all(), "| in SCC:", snap1[node_col].isin(scc).mean().round(4))


columns: ['uid', 'snap_node', 'snap_dist_m']
node column: snap_node
Central|24         425 m ->    113 m   (nodes inside=386)
East|46            414 m ->    521 m   (nodes inside=324)
South|29           279 m ->    138 m   (nodes inside=272)
East|27            463 m ->    499 m   (nodes inside=270)
East|33           1502 m ->    111 m   (nodes inside=404)
East|49            306 m ->    274 m   (nodes inside=298)
Central|14         257 m ->    265 m   (nodes inside=264)
South|37           260 m ->    121 m   (nodes inside=663)
West|51            484 m ->    318 m   (nodes inside=314)
Central|30         318 m ->    270 m   (nodes inside=176)
North|12           543 m ->    513 m   (nodes inside=749)
Central|12         370 m ->    357 m   (nodes inside=346)
North|67           486 m ->    166 m   (nodes inside=608)
East|24            256 m ->    136 m   (nodes inside=276)
North|11           272 m ->    326 m   (nodes inside=537)

rows changed: 9 (expected 15);  wrote /content/drive/MyDrive/

In [ ]:
import geopandas as gpd, osmnx as ox, pandas as pd, networkx as nx
from shapely.geometry import Point

G = ox.load_graphml(f'{ROOT}/03_network/blr_drive.graphml')
w = gpd.read_file(f'{ROOT}/02_data/wards_gba_2025_369.gpkg').to_crs(4326)
snap = pd.read_csv(f'{ROOT}/03_network/ward_snap_nodes.csv')

nodes = gpd.GeoDataFrame(
    {'n': list(G.nodes())},
    geometry=[Point(G.nodes[n]['x'], G.nodes[n]['y']) for n in G.nodes()],
    crs=4326)

bad = snap[snap.snap_dist_m > 250].uid.tolist()
for u in bad:
    poly = w[w.uid == u].geometry.iloc[0]
    inside = nodes[nodes.within(poly)]
    d = snap[snap.uid == u].snap_dist_m.iloc[0]
    print(f"{u:14} snap={d:7.0f} m  nodes_inside_ward={len(inside):5d}  "
          f"{'-> centroid artifact, FIXABLE' if len(inside) > 0 else '-> NO ROADS, real gap'}")


Central|24     snap=    425 m  nodes_inside_ward=  386  -> centroid artifact, FIXABLE
East|46        snap=    414 m  nodes_inside_ward=  326  -> centroid artifact, FIXABLE
South|29       snap=    279 m  nodes_inside_ward=  272  -> centroid artifact, FIXABLE
East|27        snap=    463 m  nodes_inside_ward=  270  -> centroid artifact, FIXABLE
East|33        snap=   1502 m  nodes_inside_ward=  404  -> centroid artifact, FIXABLE
East|49        snap=    306 m  nodes_inside_ward=  300  -> centroid artifact, FIXABLE
Central|14     snap=    257 m  nodes_inside_ward=  264  -> centroid artifact, FIXABLE
South|37       snap=    260 m  nodes_inside_ward=  666  -> centroid artifact, FIXABLE
West|51        snap=    484 m  nodes_inside_ward=  314  -> centroid artifact, FIXABLE
Central|30     snap=    318 m  nodes_inside_ward=  177  -> centroid artifact, FIXABLE
North|12       snap=    543 m  nodes_inside_ward=  749  -> centroid artifact, FIXABLE
Central|12     snap=    370 m  nodes_inside_ward=  346

### Cell 4b — Overpass fallback

Only if Cell 4 failed on a rate limit or timeout. Downloads the Geofabrik extract
instead (~700 MB) and builds the graph with `pyrosm`. Slower, but not rate
limited.

In [ ]:
# !pip install -q pyrosm
# !wget -q -O /content/southern.osm.pbf https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf
# from pyrosm import OSM
# osm = OSM('/content/southern.osm.pbf', bounding_box=[77.4599,12.8335,77.7841,13.1426])
# nodes, edges = osm.get_network(nodes=True, network_type='driving')
# G = osm.to_graph(nodes, edges, graph_type='networkx')
# import osmnx as ox; ox.save_graphml(G, f'{ROOT}/03_network/blr_drive.graphml')
print("fallback - uncomment only if Cell 4 failed")

## Cell 5 — Routed travel time, CONTROL profile

Two products, deliberately separate:
- **inter-ward matrix** -> feeds rebalancing
- **intra-ward distance** -> feeds the service rate mu

Deriving mu from the inter-ward matrix would reintroduce the exact error audit
finding F11 identified.

**`--profile control` is not optional for the first run.** It pins speed at the
placeholder's flat 24 km/h and reuses its congestion dict, so path geometry is
the *only* thing that differs from the placeholder. Without it, three variables
move at once (geometry, speed, congestion at 13:00 shifts 1.00 -> 1.26) and the
Cell 7 comparison cannot attribute the change to road geometry.

Run `--profile osm` afterwards as a **separate** experiment.

In [ ]:
%cd {ROOT}/03_network
if done(f'{RES}/ward_travel_time_369_routed_control.parquet'):
    stamp("routed control matrix exists - skipping")
else:
    !python3 travel_time.py --profile control --graph {ROOT}/03_network/blr_drive.graphml \
        --wards {ROOT}/02_data/wards_gba_2025_369.gpkg \
        --snap {ROOT}/03_network/ward_snap_nodes.csv --results {RES}
stamp("done")

/content/drive/MyDrive/qcommerce/03_network
  speed held flat at 24.0 km/h (control profile)
  routing 369 sources over 188,396 nodes...
INTER-WARD {
  "evidence_class": "ESTIMATED",
  "method": "OSM road routing + congestion scaling",
  "congestion_source": "held equal to the placeholder layer so routing is the only variable",
  "profile": "control",
  "n_wards": 369,
  "n_rows": 2308464,
  "unreachable_pairs": -369,
  "median_road_km": 13.13,
  "median_freeflow_min": 32.81
}

INTRA-WARD {
  "evidence_class": "ESTIMATED",
  "method": "uniform in-polygon sampling + road routing",
  "n_sample": 120,
  "dark_store_assumption": "ward representative point (Week 4 refines)",
  "median_mean_m": 876.1,
  "min_mean_m": 233.7,
  "max_mean_m": 4306.4
}

SERVICE RATE mu: median 6.68, range 2.07-9.53 orders/rider-hr
  V1 used a flat 6.0 (Pune) / 7.5 (Delhi) with no source — see audit F11.
  by hour (median across wards):
    09:00  5.93
    13:00  7.08
    20:00  6.02
[14:11:44] done


## Cell 6 — Population from building footprints

Two separable questions, kept separate — conflating them is what produced the
rejected layer in Week 2.

**Control total:** Census 2011 BBMP = 8,443,675. OBSERVED and citable, 14 years
stale. Carry the growth factor to ~13.6 M explicitly as a parameter; never bury
it inside a population figure.

**Weight:** building footprints beat WorldPop here. WorldPop's constrained
product is itself modelled from census + built-up area, so it is a smarter
interpolation of the same census, not independent evidence. Footprints come from
current imagery.

Caveat to state in the paper: footprint *area* is not floor area — apartments and
sheds weigh equally.

The area baseline is degenerate (density constant to 3.7e-5), so this is a
**prerequisite**, not an improvement.

In [ ]:
import os, gzip, json, math, requests
import geopandas as gpd, pandas as pd
from shapely.geometry import shape

FPDIR = f'{ROOT}/02_data/footprints'; os.makedirs(FPDIR, exist_ok=True)
OUT = f'{FPDIR}/bengaluru_buildings.geojson'
QK = '123303312'          # level-9 quadkey covering the full GBA bbox
LINKS = 'https://bfppub.blob.core.windows.net/%24web/2026-08-13/dataset-links.csv'

if os.path.exists(OUT):
    print("footprints already in Drive:", OUT)
else:
    dl = pd.read_csv(LINKS)
    print("columns:", list(dl.columns))
    qcol = [c for c in dl.columns if 'quad' in c.lower()][0]
    ucol = [c for c in dl.columns if 'url'  in c.lower()][0]
    rows = dl[dl[qcol].astype(str) == QK]
    print(f"{len(rows)} tile(s) for quadkey {QK}")
    if len(rows) == 0:
        raise SystemExit("quadkey not found - print dl[qcol].unique() and check")

    geoms = []
    for u in rows[ucol]:
        raw = f'/content/{QK}.csv.gz'
        with open(raw, 'wb') as f:
            f.write(requests.get(u, timeout=600).content)
        # extension is .csv.gz but contents are GeoJSONL, one feature per line
        with gzip.open(raw, 'rt') as f:
            for line in f:
                line = line.strip()
                if line:
                    geoms.append(shape(json.loads(line)['geometry']))
    print(f"{len(geoms):,} footprints in tile")

    b = gpd.GeoDataFrame(geometry=geoms, crs=4326)
    w = gpd.read_file(f'{ROOT}/02_data/wards_gba_2025_369.gpkg').to_crs(4326)
    b = b[b.intersects(w.union_all())]
    print(f"{len(b):,} inside the GBA boundary")
    b.to_file(OUT, driver='GeoJSON')

!cd {ROOT}/02_data && python3 population.py \
    --wards {ROOT}/02_data/wards_gba_2025_369.gpkg \
    --control census2011_bbmp --weight buildings --buildings {OUT} \
    --out {RES}/ward_population_369.csv

p = pd.read_csv(f'{RES}/ward_population_369.csv')
print("density std/mean:", round(p.density_per_km2.std()/p.density_per_km2.mean(), 4),
      "  (area baseline was 0.000037 — degenerate)")


columns: ['Location', 'QuadKey', 'Url', 'Size', 'UploadDate']
1 tile(s) for quadkey 123303312
1,063,902 footprints in tile
597,673 inside the GBA boundary
{
  "control_total": {
    "value": 8443675,
    "year": 2011,
    "evidence_class": "OBSERVED",
    "source": "Census of India 2011, Primary Census Abstract, BBMP (M Corp.)",
    "note": "Audited and citable. 14 years stale: the city has roughly 1.5x'd since. Understates ALL wards, and understates peri-urban wards differentially \u2014 which is where Q-commerce is expanding."
  },
  "weight": {
    "method": "building_footprint",
    "evidence_class": "OBSERVED (footprints)",
    "source": "/content/drive/MyDrive/qcommerce/02_data/footprints/bengaluru_buildings.geojson",
    "n_buildings": 597673,
    "note": "footprint area, not floor area; tall and low buildings weigh equally"
  },
  "n_wards": 369,
  "allocated_total": 8443675.0,
  "target_total": 8443675.0,
  "absolute_error": 0.0,
  "relative_error_pct": 0.0,
  "reconciled": tr

## Cell 7 — V2.1 EXPERIMENT: placeholder vs road-routed

**This is a publishable result on its own**, independent of everything
downstream: *how much does replacing abstract distance with road-network travel
change the optimal q-commerce network?*

The model is **frozen** — objective, store capacity, coverage threshold, integer
siting, assignment formulation and zone logic all unchanged. Only the
travel-time input differs, and Cell 5's control profile ensures that difference
is geometry alone.

Watch the **orphan_wards** column. Under routed times some wards may lose
coverage entirely, which matters more than a change in store count.

In [ ]:
%cd {ROOT}/05_network_design
!python3 experiments.py --compare \
    placeholder={RES}/ward_travel_time_369.parquet \
    routed_control={RES}/ward_travel_time_369_routed_control.parquet \
    --demand {RES}/demand_369.parquet --adjacency {RES}/ward_adjacency_369.csv --out {RES}

import pandas as pd
df = pd.read_csv(f'{RES}/experiment_travel_time_comparison.csv')
cols = [c for c in ['label','orphan_wards','stores','sites','zones','singleton_zones',
                    'mean_util','mean_minutes_excl_self','p90_minutes_excl_self',
                    'self_served_order_pct','total_cost_hr'] if c in df]
print(df[cols].to_string(index=False))

/content/drive/MyDrive/qcommerce/05_network_design

         label  cover_min  status  orphan_wards  stores  sites  zones  singleton_zones  mean_util  mean_minutes_excl_self  p90_minutes_excl_self  self_served_order_pct  total_cost_hr
   placeholder       10.0 optimal             0     286    198    208              118      0.975                    5.66                   8.81                   79.2      1429530.6
routed_control       10.0 optimal             0     295    202    214              129      0.952                    5.73                   8.74                   80.1      1465664.3

wrote /content/drive/MyDrive/qcommerce/results/experiment_travel_time_comparison.csv
         label  orphan_wards  stores  sites  zones  singleton_zones  mean_util  mean_minutes_excl_self  p90_minutes_excl_self  self_served_order_pct  total_cost_hr
   placeholder             0     286    198    208              118      0.975                    5.66                   8.81                   79.2 

### Cell 7b — Coverage sweep on routed times

Repeats the Week 4 sweep (5 / 7.5 / 10 / 12.5 / 15 min). On placeholder geometry
all five solved and store count fell only 23% across a 3x loosening — the network
was capacity-bound, not coverage-bound, beyond ~10 min. Does that survive real
roads?

In [ ]:
!python3 experiments.py --sweep --label routed_control \
    --travel {RES}/ward_travel_time_369_routed_control.parquet \
    --demand {RES}/demand_369.parquet --adjacency {RES}/ward_adjacency_369.csv --out {RES}
print(pd.read_csv(f'{RES}/experiment_coverage_sweep.csv').to_string(index=False))


--- routed_control @ cover_min=5.0 ---

--- routed_control @ cover_min=7.5 ---

--- routed_control @ cover_min=10.0 ---

--- routed_control @ cover_min=12.5 ---

--- routed_control @ cover_min=15.0 ---

         label  cover_min  status  orphan_wards  stores  sites  zones  singleton_zones  mean_util  mean_minutes_excl_self  p90_minutes_excl_self  self_served_order_pct  total_cost_hr
routed_control        5.0 optimal             0     365    243    246              178      0.727                    3.83                   4.81                   88.3      1696732.9
routed_control        7.5 optimal             0     316    209    215              132      0.883                    5.00                   7.00                   82.2      1529657.3
routed_control       10.0 optimal             0     295    202    214              129      0.952                    5.73                   8.74                   80.1      1465664.3
routed_control       12.5 optimal             0     284    196  

## Cell 8 — Demand re-run, and the GATE

**Do not proceed past this cell if the rank correlation is above ~0.95.**

That gate exists because the first demand engine produced adjacent-hour rank
correlation of 1.000 — spatial imbalance with zero temporal churn, meaning the
busy wards are the same all day and riders placed well once never need to move.
That is V1's defect reappearing in new code. Building the SMILP on such a surface
wastes the solve.

In [ ]:
import json
d = json.load(open(f'{ROOT}/04_demand/demand_diagnostics.json'))
for k,v in d['heterogeneity'].items():
    print(k, 'peakoff=',v.get('wards_peaking_off_city_peak'), 'raw=',v.get('mean_adjacent_hour_rank_corr'))
    print('   norm=',v.get('mean_adjacent_hour_rank_corr_normalised'), 'volspread=%.3g'%v.get('ward_volume_spread_x',0), 'shape=',v.get('within_ward_shape_spread_x'))
print({k:v for k,v in d['diagnostics'].items() if not isinstance(v,(dict,list))} if isinstance(d['diagnostics'],dict) else d['diagnostics'])


BAU peakoff= 0.4526 raw= 0.9905
   norm= 0.7495 volspread=5.58e+12 shape= 5.2
WEEKEND peakoff= 0.4526 raw= 0.9904
   norm= 0.7498 volspread=6.64e+12 shape= 5.2
EVENT peakoff= 0.4526 raw= 0.9927
   norm= 0.7498 volspread=1.03e+13 shape= 5.2
MEGA peakoff= 0.4526 raw= 0.9939
   norm= 0.7499 volspread=2.4e+13 shape= 5.2
{'evidence_class': 'ESTIMATED (chain of declared factors)', 'hourly_shape_source': 'ESTIMATED — twin peaks at lunch (12-13) and dinner (19-21), the established q-commerce pattern. Replace with observed platform or delivery-aggregator temporal data when available.', 'archetype_source': 'ESTIMATED — assigned by density tercile as a proxy for land use. Replace with OSM POI mix (office/retail share vs residential) when the road graph is available.', 'shape_model': 'continuous commercial/residential blend', 'commercial_share_mean': 0.5012, 'commercial_share_std': 0.2846, 'control_total_population': 8443675.0, 'daily_city_orders_BAU': 689968.9, 'weekly_city_orders_BAU': 4829782.1

## Cell 9 — Facility + SMILP re-run on real inputs

Slowest stage. Each SMILP solve took 8-10 min on this sandbox and Colab gives 2
vCPUs, so budget 15-20 min each.

If time is tight, raise the MIP gap to 0.05 — defensible, since the cost
parameters are ASSUMED and a 1% gap is false precision against them.

In [ ]:
%cd {ROOT}/05_network_design
!python3 facility_location.py --demand {RES}/demand_369.parquet \
    --travel {RES}/ward_travel_time_369_routed_control.parquet \
    --adjacency {RES}/ward_adjacency_369.csv --results {RES}

%cd {ROOT}/06_smilp_v2
!python3 smilp_v2.py --zones {RES}/zones_v2.csv --demand {RES}/demand_369.parquet \
    --zadj {RES}/zone_adjacency_v2.csv --wardprofile {RES}/ward_population_369.csv \
    --stores {RES}/dark_stores.csv --out {RES}/final_v2r.json
show(f'{RES}/final_v2r.json')

/content/drive/MyDrive/qcommerce/05_network_design
FACILITY LOCATION
{
  "status": "optimal",
  "n_stores": 452,
  "n_sites": 304,
  "n_wards": 369,
  "max_stores_at_one_site": 5,
  "planning_scenario": "EVENT",
  "planning_hour": 20,
  "cover_min": 10.0,
  "store_cap_orders_hr": 220.0,
  "peak_hour_orders": 95721.4,
  "total_cost_per_hr": 2181458.4,
  "facility_cost_per_hr": 2034000.0,
  "mean_store_utilisation": 0.97,
  "min_store_utilisation": 0.001,
  "wards_split_across_stores": 174,
  "mean_service_minutes": 0.77,
  "p90_service_minutes": 8.38,
  "evidence_class": "ESTIMATED (real geography, ASSUMED cost/capacity params)",
  "parameter_note": "All Stage-1 parameters are ASSUMED and none is load-bearing on its own \u2014 but COVER_MIN and STORE_CAP_ORDERS_HR jointly determine the store count, so they must be reported together with any network recommendation."
}

ZONE FORMATION
{
  "n_zones": 312,
  "n_stores": 303,
  "n_wards": 368,
  "fragmented_zones": 9,
  "fragmentation": {
  

### Cell 9b — Re-test the mechanism on real geometry

**The movement-economics finding is the single result most exposed to routed
travel times**, because real road distance changes relocation cost directly. On
placeholder geometry: zero flow at baseline, 160,976 flow when movement was made
free — and even free movement bought only Rs 838 and 0.00 points of service
level.

Does that hold on real roads?

In [ ]:
%cd {ROOT}/07_experiments
!python3 week8.py --cap 220 --tag real_base
!python3 week8.py --cap 220 --tag real_free --free-rebal
for t in ['real_base','real_free']:
    r = json.load(open(f'{RES}/w8_{t}.json'))
    fl = sum(v['rebal_flow'] for v in r['per_scenario'].values())
    print(f"{t:10} cost={r['total_cost']:>14,.0f} expSL={r['expected_sl']:6.2f} flow={fl:>12,.1f}")

/content/drive/MyDrive/qcommerce/07_experiments
{
  "tag": "real_base",
  "store_cap": 220.0,
  "total_cost": 60262003.3,
  "total_riders": 58747.0,
  "expected_sl": 95.685
}
   BAU       SL= 97.97  rebal_flow=       0.0
   WEEKEND   SL= 96.42  rebal_flow=       0.0
   EVENT     SL= 95.96  rebal_flow=      -0.0
   MEGA      SL= 64.12  rebal_flow=      -0.0
{
  "tag": "real_free",
  "store_cap": 220.0,
  "total_cost": 60272550.2,
  "total_riders": 58746.0,
  "expected_sl": 95.684
}
   BAU       SL= 97.97  rebal_flow=  82,283.4
   WEEKEND   SL= 96.42  rebal_flow=  73,342.0
   EVENT     SL= 95.96  rebal_flow=  57,466.1
   MEGA      SL= 64.11  rebal_flow=  16,575.7
real_base  cost=    60,262,003 expSL= 95.69 flow=         0.0
real_free  cost=    60,272,550 expSL= 95.68 flow=   229,667.2


## Cell 10 — Store-capacity costing

The project's strongest *positive* finding: quadrupling dark-store throughput
took MEGA service 48.12% -> 85.67% and halved total cost.

But the sweep shows what throughput would **buy**, not what it would **cost to
build**. Costing it converts a suggestive number into a recommendation, and it
reframes the trade-off correctly — facility vs labour, not FT vs gig.

Fill in real capex/opex per store before using any of this.

In [ ]:
import shutil, time
snap = f'{ROOT}/snapshots/{time.strftime("%Y%m%d_%H%M")}_patched'
shutil.copytree(RES, snap)
print("snapshot ->", snap)


snapshot -> /content/drive/MyDrive/qcommerce/snapshots/20261022_1901_patched


## Cell 11 — Snapshot results back to Drive

Run after any long stage. Keeps a timestamped copy so a later re-run cannot
overwrite a result you have already reported.

In [ ]:
import shutil, time
snap = f'{ROOT}/snapshots/{time.strftime("%Y%m%d_%H%M")}'
shutil.copytree(RES, snap)
print("snapshot ->", snap)

snapshot -> /content/drive/MyDrive/qcommerce/snapshots/20261022_1901


In [ ]:
from google.colab import files
import shutil, os

# Ensure ROOT is defined if this cell is run independently
try:
    ROOT
except NameError:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/qcommerce'

dest = {'travel_time.py': f'{ROOT}/03_network', 'demand_engine.py': f'{ROOT}/04_demand'}
up = files.upload()
for name, data in up.items():
    key = [k for k in dest if name.endswith(k)][0]
    target = f'{dest[key]}/{key}'
    if os.path.exists(target) and not os.path.exists(target + '.orig'):
        shutil.copy(target, target + '.orig')   # keep the pre-patch original once
    with open(target, 'wb') as f:
        f.write(data)
    print('installed', target, len(data), 'bytes | backup:', os.path.exists(target + '.orig'))

Mounted at /content/drive
